# Final evaluation (configs/final_v1.yaml)

Approved by the team on 2026-11-09. Opens the test split and the five held-out countries **once**.
Nothing is trained; every threshold comes from validation data.

1. **Check on validation:** the same pipeline on val must reproduce the reported numbers (top-1 from
   `reports/`, AUC and hierarchy scores from the saved predictions). If it does not, the notebook stops
   **before** the test split is opened.
2. **Test:** top-1 by ink on every test drawing; when to guess and the hierarchy on 500 drawings per class;
   the same on the held-out countries for the deployed models.

**Before running:** attach, besides `dataset-ai-pictionary`, the datasets that hold the checkpoints
(leak_v1, step 4a, followup_v1, followup_v2) and the saved validation predictions
(`runs/predictions_all`). Choose **GPU T4 x2**, **Internet on**, then **Save Version → Save & Run All**.


In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Trung4n/AI-dictionary.git"
BRANCH = "pipeline-step1"
REPO = Path("/kaggle/working/repo")

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    print("No GITHUB_TOKEN secret: cloning without one (works for a public repository).")

url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
os.chdir("/kaggle/working")  # a re-run may still be inside the folder deleted next
if REPO.exists():
    shutil.rmtree(REPO)
done = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO)], capture_output=True, text=True)
if done.returncode != 0:
    message = done.stderr.replace(token, "***") if token else done.stderr
    raise RuntimeError(f"clone failed:\n{message}")
# Keep the token out of files that Kaggle may save with the notebook output.
subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


In [ ]:
%cd /kaggle/working/repo
!pip install -q pytest
tests = subprocess.run("PYTHONPATH=src python -m pytest tests -q -k 'not processed_data'", shell=True, capture_output=True, text=True)
print(tests.stdout[-1500:])
if tests.returncode != 0:
    raise RuntimeError("tests failed: stopping before using GPU quota")

candidates = [p.parent for p in Path("/kaggle/input").rglob("gru_anytime_uniform_s42.npz")]
if not candidates:
    raise RuntimeError("saved validation predictions not found: attach the followup_v2 output (runs/predictions_all)")
VAL_PREDICTIONS = max(candidates, key=lambda d: len(list(d.glob("*.npz"))))
print("validation predictions:", VAL_PREDICTIONS, len(list(VAL_PREDICTIONS.glob("*.npz"))), "files")


In [ ]:
# Step 1: validation check. The test split is not opened unless this passes.
done = subprocess.run(f"PYTHONPATH=src python -m quickdraw_exp.final_eval --split val --val-predictions {VAL_PREDICTIONS}",
                      shell=True, capture_output=True, text=True)
print(done.stdout[-6000:])
print(done.stderr[-3000:])
if done.returncode != 0:
    raise RuntimeError("validation check failed: the test split was NOT opened")


In [ ]:
# Step 2: the one test run.
!PYTHONPATH=src python -m quickdraw_exp.final_eval --split test --val-predictions {VAL_PREDICTIONS}


In [ ]:
import json

check = json.loads(Path("/kaggle/working/runs/final_v1_val/results.json").read_text())["check"]
print("validation check:", check["compared"], "numbers, largest difference", check["largest"],
      "probs max abs diff", round(check["probs_max_abs_diff"], 5))
r = json.loads(Path("/kaggle/working/runs/final_v1_test/results.json").read_text())

def show(rows, label):
    print(f"\n{label}")
    for row in rows:
        tag = f"[{row['precision_target']}] " if "precision_target" in row else ""
        if row.get("difference") is None:
            print(f"  {tag}{row['name']}: not available")
        else:
            print(f"  {tag}{row['name']}: {row['a_mean']:.4f} vs {row['b_mean']:.4f}, difference {row['difference']:+.4f}  "
                  f"(a {[round(v, 4) for v in row['a_values']]}, b {[round(v, 4) for v in row['b_values']]})")

for name, s in r["sets"].items():
    print(f"\n==================== {name}: {s['drawings']} drawings, {s['sampled_drawings']} sampled ====================")
    print("Top-1 by ink fraction:")
    for run, by in s["leak"]["per_run"].items():
        print(f"  {run:24s}", {f: round(v["top1"], 4) for f, v in by.items()})
    show(s["leak"]["comparisons"], "Early top-1 (10, 25, 50% ink):")
    show(s["anytime"]["comparisons"], "When to guess (AUC):")
    print("\nPer run: AUC (pmax / conf), risk control at 2% and 5%:")
    for run, rep in s["anytime"]["per_run"].items():
        rc = rep["pmax"]["risk_control"]
        conf = f" conf {rep['conf']['auc']:.4f}" if "conf" in rep else ""
        print(f"  {run:24s} pmax {rep['pmax']['auc']:.4f}{conf}  gap2% {rc['0.02']['gap']:.4f} met {rc['0.02']['met']} "
              f"earliness {rc['0.02']['earliness']:.3f}  gap5% {rc['0.05']['gap']:.4f} met {rc['0.05']['met']} "
              f"earliness {rc['0.05']['earliness']:.3f}  end {rep['reference']['end_of_drawing']['accuracy']:.4f}")
    show(s["hierarchy"]["comparisons"], "Hierarchy (reward area):")
